In [1]:
import pandas as pd

In [2]:
df21_22 = pd.read_csv('dataset/PL21:22.csv')
df22_23 = pd.read_csv('dataset/PL22:23.csv')
df23_24 = pd.read_csv('dataset/PL23:24.csv')
df24_25 = pd.read_csv('dataset/PL24:25.csv')
df25_26 = pd.read_csv('dataset/PL25:26.csv')

df21_22.insert(0, 'Season', '2021/22')
df22_23.insert(0, 'Season', '2022/23')
df23_24.insert(0, 'Season', '2023/24')
df24_25.insert(0, 'Season', '2024/25')
df25_26.insert(0, 'Season', '2025/26')

df = pd.concat([df21_22, df22_23, df23_24, df24_25, df25_26])
df = df.sort_values(by='Date',ascending=True)
df = df.reset_index(drop=True)
df

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
0,2021/22,2021-08-13,Brentford,Arsenal,2,0,H,1,0,H,...,3,4,12,8,2,5,0,0,0,0
1,2021/22,2021-08-14,Man United,Leeds,5,1,H,1,0,H,...,8,3,11,9,5,4,1,2,0,0
2,2021/22,2021-08-14,Burnley,Brighton,1,2,A,1,0,H,...,3,8,10,7,7,6,2,1,0,0
3,2021/22,2021-08-14,Chelsea,Crystal Palace,3,0,H,2,0,H,...,6,1,15,11,5,2,0,0,0,0
4,2021/22,2021-08-14,Everton,Southampton,3,1,H,0,1,A,...,6,3,13,15,6,8,2,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1874,2025/26,2026-05-10,Nott'm Forest,Newcastle,1,1,D,0,0,D,...,6,6,16,11,2,1,2,0,0,0
1875,2025/26,2026-05-10,West Ham,Arsenal,0,1,A,0,0,D,...,3,4,14,12,4,3,3,4,0,0
1876,2025/26,2026-05-10,Burnley,Aston Villa,2,2,D,1,1,D,...,6,7,18,8,2,8,1,1,0,0
1877,2025/26,2026-05-10,Crystal Palace,Everton,2,2,D,1,1,D,...,8,6,5,13,5,10,0,2,0,0


In [3]:
df.isna().sum()

Season      0
Date        0
HomeTeam    0
AwayTeam    0
FTHG        0
FTAG        0
FTR         0
HTHG        0
HTAG        0
HTR         0
Referee     0
HS          0
AS          0
HST         0
AST         0
HF          0
AF          0
HC          0
AC          0
HY          0
AY          0
HR          0
AR          0
dtype: int64

In [4]:
df = df.drop(columns='Referee')

In [5]:
def get_last_5_games(team, index):
    return df.iloc[:index][
        (df.iloc[:index]['HomeTeam'] == team) | (df.iloc[:index]['AwayTeam'] == team)].iloc[-5:]

def get_last_5_home_games(team, index):
    return df.iloc[:index][(df.iloc[:index]['HomeTeam'] == team)].iloc[-5:]
    
def get_last_5_away_games(team, index):
    return df.iloc[:index][(df.iloc[:index]['AwayTeam'] == team)].iloc[-5:]

get_last_5_games('Man City', -1)

,Season,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,...,HST,AST,HF,AF,HC,AC,HY,AY,HR,AR
1837,2025/26,2026-04-12,Chelsea,Man City,0,3,A,0,0,D,...,3,8,10,14,4,12,3,1,0,0
1847,2025/26,2026-04-19,Man City,Arsenal,2,1,H,1,1,D,...,5,3,5,12,8,5,2,2,0,0
1851,2025/26,2026-04-22,Burnley,Man City,0,1,A,0,1,A,...,1,9,10,12,3,11,0,0,0,0
1868,2025/26,2026-05-04,Everton,Man City,3,3,D,0,1,A,...,6,4,15,5,5,9,4,1,0,0
1871,2025/26,2026-05-09,Man City,Brentford,3,0,H,0,0,D,...,10,2,8,14,10,2,4,2,0,0


In [6]:
def get_form(team, current_index):

    form = []

    matches = get_last_5_games(team, current_index)
    
    for i in range(matches.shape[0]):

        home_team = matches.iloc[i]['HomeTeam']
        away_team = matches.iloc[i]['AwayTeam']
        result = matches.iloc[i]['FTR']

        if home_team == team:

            if result == 'H':
                form.append('W')

            elif result == 'D':
                form.append('D')

            elif result == 'A':
                form.append('L')

        elif away_team == team:

            if result == 'A':
                form.append('W')

            elif result == 'D':
                form.append('D')

            elif result == 'H':
                form.append('L')

    form.reverse()

    return form

get_form('Man City', -1)

['W', 'D', 'W', 'W', 'W']

In [7]:
def get_form_points(team, index) :
    form = get_form(team, index)
    points = 0
    for i in form:
        if i == 'W':
            points += 3
        elif i == 'D':
            points += 1
    
    return points
get_form_points('Man City', -1)

13

In [8]:
def get_avg_goals_scored(team, index):
    matches = get_last_5_games(team, index)
    goals = 0
    
    for i in range(matches.shape[0]):
        if matches.iloc[i]['HomeTeam'] == team:
            goals += matches.iloc[i]['FTHG']
        else:
            goals += matches.iloc[i]['FTAG']
    
    average = goals/5
    return average

get_avg_goals_scored('Man City', -1)

np.float64(2.4)

In [9]:
def get_avg_goals_conceeded(team, index):
    matches = get_last_5_games(team, index)
    goals_conceeded = 0
    
    for i in range(matches.shape[0]):
        if matches.iloc[i]['HomeTeam'] == team:
            goals_conceeded += matches.iloc[i]['FTAG']
        else:
            goals_conceeded += matches.iloc[i]['FTHG']
            
    avg = goals_conceeded/5
    
    return avg

get_avg_goals_conceeded('Man City', -1)

np.float64(0.8)

In [10]:
def get_avg_GD(team, index):
    gs = get_avg_goals_scored(team, index)
    gc = get_avg_goals_conceeded(team, index)
    
    gd = gs-gc
    
    return round(gd, 2)
get_avg_GD('Man City', -1)

np.float64(1.6)

In [11]:
def get_home_strength(team, index):
    games = get_last_5_home_games(team, index)
    strength = 0
    
    for i in range(games.shape[0]):
        if games.iloc[i]['FTR'] == 'H':
            strength += 3
        
        elif games.iloc[i]['FTR'] == 'D':
            strength += 1
    
    return strength

get_home_strength('Man City', -1)

13

In [12]:
def get_away_strength(team, index):
    games = get_last_5_away_games(team, index)
    strength = 0
    
    for i in range(games.shape[0]):
        if games.iloc[i]['FTR'] == 'A':
            strength += 3
        
        elif games.iloc[i]['FTR'] == 'D':
            strength += 1
    
    return strength

get_away_strength('Man City', -1)


11

In [13]:
def points_per_game(team, index) :
    
    points = 0
    season = df.iloc[index]['Season']
    games = df.iloc[:index]
    games = games[games['Season'] == season]
    team_games = 0
    
    for i in range(games.shape[0]):
        home = games.iloc[i]['HomeTeam']
        away = games.iloc[i]['AwayTeam']
        result = games.iloc[i]['FTR']
        
        if home == team:
            if result == 'H':
                points += 3
                team_games += 1
            elif result == 'D':
                points += 1
                team_games += 1
            else:
                team_games += 1
                
        elif away == team:
            if result == 'A':
                points += 3
                team_games += 1
            elif result == 'D':
                points += 1
                team_games += 1
            else:
                team_games += 1
                
    if team_games == 0:
        return 0
    
    avg = points/team_games
    return round(avg,1)

points_per_game('Liverpool',-360)

2.2

In [14]:
def make_dataset():
    
    rows = []
    
    for i in range(50, df.shape[0]):
        home_team = df.iloc[i]['HomeTeam']
        away_team = df.iloc[i]['AwayTeam']
        
        home_team_form = get_form_points(home_team, i)
        away_team_form = get_form_points(away_team, i)
        
        home_team_avg_goals = get_avg_goals_scored(home_team, i)
        away_team_avg_goals = get_avg_goals_scored(away_team, i)
        
        home_team_avg_goals_conceeded = get_avg_goals_conceeded(home_team, i)
        away_team_avg_goals_conceeded = get_avg_goals_conceeded(away_team, i)
        
        home_team_avg_GD = get_avg_GD(home_team, i)
        away_team_avg_GD = get_avg_GD(away_team, i)
        
        home_team_strength = get_home_strength(home_team, i)
        away_team_strength = get_away_strength(away_team, i)
        
        ht_ppg = points_per_game(home_team, i)
        at_ppg = points_per_game(away_team, i)
        
        result = df.iloc[i]['FTR']
        
        row = {
            'Home Team' : home_team,
            'Away Team' : away_team,
            'Home Team Form' : home_team_form,
            'Away Team Form' : away_team_form,
            'HT Avg Goals Scored' : home_team_avg_goals,
            'AT Avg Goals Scored' : away_team_avg_goals,
            'HT Avg Goals Conceeded' : home_team_avg_goals_conceeded,
            'AT Avg Goals Conceeded' : away_team_avg_goals_conceeded,
            'HT Avg GD' : home_team_avg_GD,
            'AT Avg GD' : away_team_avg_GD,
            'HT Strength' : home_team_strength,
            'AT Strength' : away_team_strength,
            'HT Points/Game' : ht_ppg,
            'AT Points/Game' : at_ppg,
            'Result' : result,
        }
        rows.append(row)
    
    dataset = pd.DataFrame(rows)
        
    return dataset

dataset = make_dataset()
dataset        

,Home Team,Away Team,Home Team Form,Away Team Form,HT Avg Goals Scored,AT Avg Goals Scored,HT Avg Goals Conceeded,AT Avg Goals Conceeded,HT Avg GD,AT Avg GD,HT Strength,AT Strength,HT Points/Game,AT Points/Game,Result
0,Watford,Newcastle,6,2,1.2,1.2,1.6,2.6,-0.4,-1.4,3,0,1.2,0.4,D
1,Chelsea,Man City,13,10,2.4,2.2,0.2,0.2,2.2,2.0,6,3,2.6,2.0,A
2,Man United,Aston Villa,13,7,2.6,1.6,0.8,1.4,1.8,0.2,6,0,2.6,1.4,A
3,Everton,Norwich,10,0,2.0,0.4,1.4,2.8,0.6,-2.4,6,0,2.0,0.0,H
4,Leeds,West Ham,3,8,1.0,2.2,2.4,1.4,-1.4,0.8,1,4,0.6,1.6,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824,Nott'm Forest,Newcastle,13,3,3.2,1.2,0.6,1.6,2.6,-0.4,6,6,1.2,1.3,D
1825,West Ham,Arsenal,7,9,1.2,1.6,1.2,0.8,0.0,0.8,9,8,1.0,2.2,A
1826,Burnley,Aston Villa,0,7,0.6,1.6,2.6,1.4,-2.0,0.2,1,2,0.6,1.7,D
1827,Crystal Palace,Everton,5,5,0.6,2.0,1.4,1.8,-0.8,0.2,8,7,1.3,1.4,D


In [15]:
dataset['Result'] = dataset['Result'].map({
    'H':0, 'D':1, 'A':2
})
dataset

,Home Team,Away Team,Home Team Form,Away Team Form,HT Avg Goals Scored,AT Avg Goals Scored,HT Avg Goals Conceeded,AT Avg Goals Conceeded,HT Avg GD,AT Avg GD,HT Strength,AT Strength,HT Points/Game,AT Points/Game,Result
0,Watford,Newcastle,6,2,1.2,1.2,1.6,2.6,-0.4,-1.4,3,0,1.2,0.4,1
1,Chelsea,Man City,13,10,2.4,2.2,0.2,0.2,2.2,2.0,6,3,2.6,2.0,2
2,Man United,Aston Villa,13,7,2.6,1.6,0.8,1.4,1.8,0.2,6,0,2.6,1.4,2
3,Everton,Norwich,10,0,2.0,0.4,1.4,2.8,0.6,-2.4,6,0,2.0,0.0,0
4,Leeds,West Ham,3,8,1.0,2.2,2.4,1.4,-1.4,0.8,1,4,0.6,1.6,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824,Nott'm Forest,Newcastle,13,3,3.2,1.2,0.6,1.6,2.6,-0.4,6,6,1.2,1.3,1
1825,West Ham,Arsenal,7,9,1.2,1.6,1.2,0.8,0.0,0.8,9,8,1.0,2.2,2
1826,Burnley,Aston Villa,0,7,0.6,1.6,2.6,1.4,-2.0,0.2,1,2,0.6,1.7,1
1827,Crystal Palace,Everton,5,5,0.6,2.0,1.4,1.8,-0.8,0.2,8,7,1.3,1.4,1


In [16]:
from sklearn.preprocessing import LabelEncoder

team_encoder = LabelEncoder()

all_teams = pd.concat([dataset['Home Team'], dataset['Away Team']])
team_encoder.fit(all_teams)

dataset['Home Team'] = team_encoder.transform(dataset['Home Team'])
dataset['Away Team'] = team_encoder.transform(dataset['Away Team'])

In [17]:
dataset

,Home Team,Away Team,Home Team Form,Away Team Form,HT Avg Goals Scored,AT Avg Goals Scored,HT Avg Goals Conceeded,AT Avg Goals Conceeded,HT Avg GD,AT Avg GD,HT Strength,AT Strength,HT Points/Game,AT Points/Game,Result
0,24,17,6,2,1.2,1.2,1.6,2.6,-0.4,-1.4,3,0,1.2,0.4,1
1,6,15,13,10,2.4,2.2,0.2,0.2,2.2,2.0,6,3,2.6,2.0,2
2,16,1,13,7,2.6,1.6,0.8,1.4,1.8,0.2,6,0,2.6,1.4,2
3,8,18,10,0,2.0,0.4,1.4,2.8,0.6,-2.4,6,0,2.0,0.0,0
4,11,25,3,8,1.0,2.2,2.4,1.4,-1.4,0.8,1,4,0.6,1.6,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1824,19,17,13,3,3.2,1.2,0.6,1.6,2.6,-0.4,6,6,1.2,1.3,1
1825,25,0,7,9,1.2,1.6,1.2,0.8,0.0,0.8,9,8,1.0,2.2,2
1826,5,1,0,7,0.6,1.6,2.6,1.4,-2.0,0.2,1,2,0.6,1.7,1
1827,7,8,5,5,0.6,2.0,1.4,1.8,-0.8,0.2,8,7,1.3,1.4,1


In [18]:
from sklearn.linear_model import LogisticRegression
LogReg = LogisticRegression()
X = dataset.drop(columns='Result')
y = dataset['Result']

In [19]:
split = round(dataset.shape[0]*0.8)

training_features = X.iloc[:split]
training_labels = y.iloc[:split]

testing_features = X.iloc[split:]
testing_labels = y.iloc[split:]

In [20]:
from sklearn.model_selection import GridSearchCV
parameters = {
    'C' : [0.1, 0.3, 0.5, 1, 2, 5, 10, 20, 100],
    'max_iter' : [500, 1000, 1500, 2000],
}

LR = GridSearchCV(LogReg, cv=5, param_grid=parameters, scoring='accuracy')
LR.fit(training_features, training_labels)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.1, 0.3, ...], 'max_iter': [500, 1000, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'accuracy'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2 : the score is also displayed;- >3 : the fol

In [21]:
LR.best_params_

{'C': 0.5, 'max_iter': 500}

In [22]:
LR.best_score_

np.float64(0.5304478937771752)

In [23]:
from sklearn.metrics import accuracy_score
predictions = LR.predict(testing_features)
accuracy_score(testing_labels, predictions)

0.4781420765027322

Input

In [43]:
home_team = input('Home Team: ')
away_team = input('Away Team: ')
print(home_team, away_team)
home = team_encoder.transform([home_team])[0]
away = team_encoder.transform([away_team])[0]

Liverpool Brentford


In [44]:
home_form = get_form_points(home_team,-1)
away_form = get_form_points(away_team,-1)
ht_avg_goals = get_avg_goals_scored(home_team,-1)
at_avg_goals = get_avg_goals_scored(away_team,-1)
ht_avg_goals_conceeded = get_avg_goals_conceeded(home_team,-1)
at_avg_goals_conceeded = get_avg_goals_conceeded(away_team,-1)
ht_gd = get_avg_GD(home_team, -1)
at_gd = get_avg_GD(away_team, -1)
ht_strength = get_home_strength(home_team,-1)
at_strength = get_away_strength(away_team,-1)
ht_ppg = points_per_game(home_team,-1)
at_ppg = points_per_game(away_team,-1)
match = [[home, away, home_form, away_form, ht_avg_goals, at_avg_goals, ht_avg_goals_conceeded, at_avg_goals_conceeded,
          ht_gd, at_gd, ht_strength, at_strength, ht_ppg, at_ppg]]


In [45]:
model = LR.best_estimator_
match_result = model.predict_proba(match)[0]

print(home_team, 'Vs', away_team)
print(f"\n{home_team} Win % : {match_result[0]*100:.2f}%")
print(f"Draw % : {match_result[1]*100:.2f}%")
print(f"{away_team} Win % : {match_result[2]*100:.2f}%")

Liverpool Vs Brentford

Liverpool Win % : 48.97%
Draw % : 24.32%
Brentford Win % : 26.71%


/Users/kushbajpai/Documents/Code/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(


In [27]:
from xgboost import XGBClassifier
xgboost = XGBClassifier(
    max_depth=3,
    n_estimators=100,
    learning_rate=0.05,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=3,
    random_state=42
)
xgboost.fit(training_features, training_labels)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes fr

In [28]:
xgb_predictions = xgboost.predict(testing_features)
accuracy_score(testing_labels,xgb_predictions)

0.4644808743169399

In [49]:
result = xgboost.predict_proba(match)[0]

print(home_team, 'Vs', away_team)
print(f"\n{home_team} Win % : {result[0]*100:.2f}%")
print(f"Draw % : {result[1]*100:.2f}%")
print(f"{away_team} Win % : {result[2]*100:.2f}%")

Liverpool Vs Brentford

Liverpool Win % : 57.43%
Draw % : 18.65%
Brentford Win % : 23.91%
